In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df.show(10)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pivoted_df = (
    df.groupBy("Motor energy")
      .pivot("TIME_PERIOD")
      .agg(F.round(F.sum("registrations") / 1000, 1))
      .fillna(0)
      .orderBy("Motor energy")
)

raw_rows = pivoted_df.collect()
columns = pivoted_df.columns
years = sorted([c for c in columns if c != "Motor energy"])

matrix_dict = {
    r["Motor energy"]: [r[y] for y in years]
    for r in raw_rows
}

heatmap_data = pd.DataFrame.from_dict(matrix_dict, orient="index", columns=years)

plt.figure(figsize=(12, 6))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="",
    annot_kws={"weight": "bold", "size": 9},
    cmap="YlOrRd",
    linewidths=0.5,
    linecolor="#2c3e50",
    cbar_kws={'label': 'Registrations (in Thousands)'}
)

# Heatmap border
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.5)

plt.title("Total Car Registrations (in Thousands) by TIME_PERIOD and Motor energy [EU27_2020]", fontsize=14, pad=15)
plt.xlabel("TIME_PERIOD")
plt.ylabel("Motor energy category")
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig("img/powertrain_year_heatmap.png", dpi=300)
plt.show()

In [ ]:
from pyspark.sql import functions as F

numeric_types = ["IntegerType", "DoubleType", "FloatType", "LongType", "ShortType", "DecimalType"]
num_cols = [f.name for f in df.schema.fields if str(f.dataType).split("(")[0] in numeric_types]

num_summary = (
    df.select(num_cols)
    .summary("count", "mean", "stddev", "min", "50%", "max")
    .toPandas()
    .set_index("summary")
    .T
)

num_summary = num_summary.rename(columns={"50%": "median"}).astype(float).round(2)

display(num_summary)

manufacturer_col = "manufacturer_name_eu_standard_denomination"
commercial_name_col = "commercial_name"

distinct_summary = df.select(
    F.countDistinct(manufacturer_col).alias("manufacturer_unique_count"),
    F.countDistinct(commercial_name_col).alias("commercial_name_unique_count")
).toPandas()

display(distinct_summary)


energy_col = "Motor energy"

energy_counts = (
    df.groupBy(energy_col)
    .agg(F.count("*").alias("count"))
    .withColumn("percentage (%)", F.round((F.col("count") / df.count()) * 100, 2))
    .orderBy(F.col("count").desc())
    .toPandas()
)

display(energy_counts)

In [ ]:
def print_latex_table(
    df, caption, label, font_size="\\tiny", column_format=None, include_index=False
):
    temp_df = df.copy()

    if include_index:
        index_name = (
            temp_df.index.name if temp_df.index.name else "Variable / Attribute"
        )
        temp_df = temp_df.reset_index()
        temp_df = temp_df.rename(columns={"index": index_name, "summary": index_name})

    temp_df.columns = [f"{col}" for col in temp_df.columns]

    tabular_str = temp_df.to_latex(
        index=False,
        escape=True,
        float_format="%.2f",
        column_format=column_format,
        bold_rows=False,
    )

    tabular_str = (
        tabular_str.replace("\\toprule", "\\hline")
        .replace("\\midrule", "\\hline")
        .replace("\\bottomrule", "\\hline")
    )

    latex_output = f"""\\begin{{table}}[h]
\\centering
{font_size}
\\caption{{{caption}}}
\\label{{{label}}}
{tabular_str.strip()}
\\end{{table}}"""

    print(latex_output)

In [ ]:
print_latex_table(
    df=num_summary,
    caption="Summary statistics for numerical attributes in the dataset.",
    label="tab:numerical_summary",
    font_size="\\tiny",
    column_format="p{4.5cm} c c c c c c",
    include_index=True,
)

print_latex_table(
    df=distinct_summary,
    caption="Unique counts for high-cardinality categorical attributes.",
    label="tab:distinct_counts",
    font_size="\\tiny",
    column_format="p{6cm} p{6cm}",
    include_index=False,
)

print_latex_table(
    df=energy_counts,
    caption="Distribution of newly registered passenger cars by motor energy category.",
    label="tab:motor_energy_distribution",
    font_size="\\tiny",
    column_format="p{5cm} r r",
    include_index=False,
)

In [ ]:
df.show(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr_cols = [
    "TIME_PERIOD", 
    "registrations", 
    "mass_in_running_order (kg)", 
    "co2_emissions_WLTP (g/km)", 
    "engine_capacity (cm3)", 
    "engine_power (KW)", 
    "electric_energy_consumption (Wh/km)"
]

correlation_matrix = df.select(corr_cols).toPandas().corr()

plt.figure(figsize=(8, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.82}
)

plt.tight_layout()
plt.savefig("img/correlation.png", dpi=300)
plt.show()